In [1]:
"""
LSTM Model Training - Fast Version
Boluo Watershed Streamflow Prediction - PyTorch
"""

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

#==============================================================================
# 1. Evaluation Metrics
#==============================================================================
def calc_nse(observed, predicted):
    return 1 - np.sum((observed - predicted)**2) / np.sum((observed - np.mean(observed))**2)

def calc_kge(observed, predicted):
    r = np.corrcoef(observed, predicted)[0, 1]
    alpha = np.std(predicted) / np.std(observed)
    beta = np.mean(predicted) / np.mean(observed)
    return 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)

def evaluate_model(observed, predicted, print_results=True):
    r2 = np.corrcoef(observed, predicted)[0, 1]**2
    nse = calc_nse(observed, predicted)
    kge = calc_kge(observed, predicted)
    rmse = np.sqrt(mean_squared_error(observed, predicted))
    mae = mean_absolute_error(observed, predicted)
    
    if print_results:
        print(f"  R²:   {r2:.4f}")
        print(f"  NSE:  {nse:.4f}")
        print(f"  KGE:  {kge:.4f}")
        print(f"  RMSE: {rmse:.2f} m³/s")
        print(f"  MAE:  {mae:.2f} m³/s")
    
    return {'R2': r2, 'NSE': nse, 'KGE': kge, 'RMSE': rmse, 'MAE': mae}

#==============================================================================
# 2. Data Loading
#==============================================================================
def load_and_prepare_data(train_path, test_path):
    train_df = pd.read_csv(train_path, encoding='utf-8')
    test_df = pd.read_csv(test_path, encoding='utf-8')
    
    cols = train_df.columns.tolist()
    target_col = cols[-2]
    date_col = cols[-1]
    feature_cols = cols[:-2]
    
    print(f"特征数量: {len(feature_cols)}")
    print(f"训练集样本: {len(train_df)}, 测试集样本: {len(test_df)}")
    
    X_train = train_df[feature_cols].values.astype(np.float32)
    y_train = train_df[target_col].values.astype(np.float32)
    X_test = test_df[feature_cols].values.astype(np.float32)
    y_test = test_df[target_col].values.astype(np.float32)
    test_dates = test_df[date_col].values
    
    scaler_X = MinMaxScaler()
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)
    
    scaler_y = MinMaxScaler()
    y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
    y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).flatten()
    
    n_lag = 7
    n_features_per_lag = len(feature_cols) // n_lag
    
    X_train_reshaped = X_train_scaled.reshape(-1, n_lag, n_features_per_lag).astype(np.float32)
    X_test_reshaped = X_test_scaled.reshape(-1, n_lag, n_features_per_lag).astype(np.float32)
    
    return {
        'X_train': X_train_reshaped,
        'y_train': y_train_scaled.astype(np.float32),
        'X_test': X_test_reshaped,
        'y_test': y_test_scaled.astype(np.float32),
        'y_train_original': y_train,
        'y_test_original': y_test,
        'test_dates': test_dates,
        'scaler_X': scaler_X,
        'scaler_y': scaler_y
    }

#==============================================================================
# 3. LSTM Model
#==============================================================================
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_sizes=[64, 32], dropout=0.2, bidirectional=False):
        super(LSTMModel, self).__init__()
        
        self.num_directions = 2 if bidirectional else 1
        
        self.lstm1 = nn.LSTM(input_size, hidden_sizes[0], batch_first=True, bidirectional=bidirectional)
        self.dropout1 = nn.Dropout(dropout)
        
        lstm1_out = hidden_sizes[0] * self.num_directions
        self.lstm2 = nn.LSTM(lstm1_out, hidden_sizes[1], batch_first=True, bidirectional=bidirectional)
        self.dropout2 = nn.Dropout(dropout)
        
        lstm2_out = hidden_sizes[1] * self.num_directions
        self.fc1 = nn.Linear(lstm2_out, 32)
        self.bn1 = nn.BatchNorm1d(32)
        self.fc2 = nn.Linear(32, 1)
        
    def forward(self, x):
        out, _ = self.lstm1(x)
        out = self.dropout1(out)
        out, _ = self.lstm2(out)
        out = self.dropout2(out)
        out = out[:, -1, :]
        out = torch.relu(self.bn1(self.fc1(out)))
        out = self.fc2(out)
        return out

#==============================================================================
# 4. NSE Loss
#==============================================================================
class NSELoss(nn.Module):
    def __init__(self):
        super(NSELoss, self).__init__()
        
    def forward(self, pred, target):
        # NSE = 1 - SS_res / SS_tot
        # Loss = 1 - NSE = SS_res / SS_tot
        ss_res = torch.sum((target - pred) ** 2)
        ss_tot = torch.sum((target - torch.mean(target)) ** 2)
        return ss_res / (ss_tot + 1e-8)

#==============================================================================
# 5. Fast Training Function
#==============================================================================
def train_single_model(X_train, y_train, X_val, y_val, scaler_y, params):
    """快速训练单个模型"""
    
    X_train_tensor = torch.FloatTensor(X_train).to(device)
    y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1).to(device)
    X_val_tensor = torch.FloatTensor(X_val).to(device)
    
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=params['batch_size'], shuffle=True)
    
    model = LSTMModel(
        input_size=X_train.shape[2],
        hidden_sizes=[params['hidden_size1'], params['hidden_size2']],
        dropout=params['dropout'],
        bidirectional=params.get('bidirectional', False)
    ).to(device)
    
    criterion = NSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])
    
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(params['epochs']):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
        
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_tensor).cpu().numpy().flatten()
        
        val_pred_inv = scaler_y.inverse_transform(val_pred.reshape(-1, 1)).flatten()
        val_true_inv = scaler_y.inverse_transform(y_val.reshape(-1, 1)).flatten()
        val_rmse = np.sqrt(mean_squared_error(val_true_inv, val_pred_inv))
        
        if val_rmse < best_val_loss:
            best_val_loss = val_rmse
            patience_counter = 0
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= params['patience']:
                break
    
    model.load_state_dict(best_model_state)
    model.eval()
    
    with torch.no_grad():
        y_val_pred = model(X_val_tensor).cpu().numpy().flatten()
    
    y_val_pred_inv = scaler_y.inverse_transform(y_val_pred.reshape(-1, 1)).flatten()
    y_val_true_inv = scaler_y.inverse_transform(y_val.reshape(-1, 1)).flatten()
    
    metrics = evaluate_model(y_val_true_inv, y_val_pred_inv, print_results=False)
    
    return model, metrics

#==============================================================================
# 6. Fast Hyperparameter Search
#==============================================================================
def hyperparameter_search(data, n_splits=3):
    """快速超参数搜索"""
    
    print("\n" + "="*60)
    print("Fast Hyperparameter Search (3-fold CV)")
    print("="*60)
    
    # 精简参数组合
    param_combinations = [
        {'hidden_size1': 64, 'hidden_size2': 32, 'dropout': 0.2, 'learning_rate': 0.001, 'batch_size': 64, 'bidirectional': False},
        {'hidden_size1': 128, 'hidden_size2': 64, 'dropout': 0.2, 'learning_rate': 0.001, 'batch_size': 64, 'bidirectional': False},
        {'hidden_size1': 128, 'hidden_size2': 64, 'dropout': 0.1, 'learning_rate': 0.001, 'batch_size': 64, 'bidirectional': False},
        {'hidden_size1': 128, 'hidden_size2': 64, 'dropout': 0.2, 'learning_rate': 0.0005, 'batch_size': 64, 'bidirectional': False},
        {'hidden_size1': 256, 'hidden_size2': 128, 'dropout': 0.2, 'learning_rate': 0.001, 'batch_size': 64, 'bidirectional': False},
        {'hidden_size1': 128, 'hidden_size2': 64, 'dropout': 0.2, 'learning_rate': 0.001, 'batch_size': 64, 'bidirectional': True},
    ]
    
    fixed_params = {'epochs': 80, 'patience': 15}
    tscv = TimeSeriesSplit(n_splits=n_splits)
    
    results = []
    best_kge = -np.inf
    best_params = None
    
    print(f"\n测试 {len(param_combinations)} 种参数组合, {n_splits} 折CV")
    print("-"*60)
    
    for i, params in enumerate(param_combinations):
        params.update(fixed_params)
        
        print(f"\n[{i+1}/{len(param_combinations)}] hidden=[{params['hidden_size1']},{params['hidden_size2']}], "
              f"dr={params['dropout']}, lr={params['learning_rate']}, bidir={params['bidirectional']}")
        
        cv_metrics = {'KGE': [], 'NSE': [], 'RMSE': []}
        
        for fold, (train_idx, val_idx) in enumerate(tscv.split(data['X_train'])):
            _, metrics = train_single_model(
                data['X_train'][train_idx], data['y_train'][train_idx],
                data['X_train'][val_idx], data['y_train'][val_idx],
                data['scaler_y'], params
            )
            for key in cv_metrics:
                cv_metrics[key].append(metrics[key])
        
        mean_kge = np.mean(cv_metrics['KGE'])
        mean_nse = np.mean(cv_metrics['NSE'])
        mean_rmse = np.mean(cv_metrics['RMSE'])
        
        print(f"    CV: KGE={mean_kge:.4f}, NSE={mean_nse:.4f}, RMSE={mean_rmse:.2f}")
        
        results.append({'params': params.copy(), 'mean_KGE': mean_kge, 'mean_NSE': mean_nse, 'mean_RMSE': mean_rmse})
        
        if mean_kge > best_kge:
            best_kge = mean_kge
            best_params = params.copy()
            print(f"    *** New best KGE: {best_kge:.4f} ***")
    
    print(f"\n最佳参数 (KGE={best_kge:.4f}):")
    for k, v in best_params.items():
        print(f"  {k}: {v}")
    
    # 保存结果
    pd.DataFrame(results).to_csv('lstm_hyperparam_search_results.csv', index=False)
    
    return best_params, results

#==============================================================================
# 7. Final Model Training
#==============================================================================
def train_final_model(data, params, output_dir='./'):
    """训练最终模型"""
    
    print("\n" + "="*60)
    print("Training Final Model")
    print("="*60)
    
    X_train_tensor = torch.FloatTensor(data['X_train']).to(device)
    y_train_tensor = torch.FloatTensor(data['y_train']).unsqueeze(1).to(device)
    X_test_tensor = torch.FloatTensor(data['X_test']).to(device)
    
    val_size = int(0.1 * len(X_train_tensor))
    X_tr, X_val = X_train_tensor[:-val_size], X_train_tensor[-val_size:]
    y_tr, y_val = y_train_tensor[:-val_size], y_train_tensor[-val_size:]
    
    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=params['batch_size'], shuffle=True)
    
    model = LSTMModel(
        input_size=data['X_train'].shape[2],
        hidden_sizes=[params['hidden_size1'], params['hidden_size2']],
        dropout=params['dropout'],
        bidirectional=params.get('bidirectional', False)
    ).to(device)
    
    print(model)
    print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")
    
    criterion = NSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=15)
    
    print(f"\nTraining: epochs={params['epochs']}, patience={params['patience']}")
    print("-"*60)
    
    best_val_loss = float('inf')
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': []}
    
    for epoch in range(params['epochs']):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
        train_loss /= len(train_loader.dataset)
        
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val), y_val).item()
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        
        scheduler.step(val_loss)
        
        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1:3d}/{params['epochs']}] - Train: {train_loss:.6f} - Val: {val_loss:.6f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), f'{output_dir}lstm_best_model.pth')
        else:
            patience_counter += 1
            if patience_counter >= params['patience']:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break
    
    model.load_state_dict(torch.load(f'{output_dir}lstm_best_model.pth'))
    model.eval()
    
    with torch.no_grad():
        y_test_pred = model(X_test_tensor).cpu().numpy().flatten()
        y_train_pred = model(X_train_tensor).cpu().numpy().flatten()
    
    y_test_pred = data['scaler_y'].inverse_transform(y_test_pred.reshape(-1, 1)).flatten()
    y_train_pred = data['scaler_y'].inverse_transform(y_train_pred.reshape(-1, 1)).flatten()
    
    print("\n" + "="*40)
    print("Test Set:")
    test_metrics = evaluate_model(data['y_test_original'], y_test_pred)
    
    print("\nTraining Set:")
    train_metrics = evaluate_model(data['y_train_original'], y_train_pred)
    
    # 保存
    torch.save(model.state_dict(), f'{output_dir}lstm_final_model.pth')
    joblib.dump(data['scaler_X'], f'{output_dir}scaler_X.pkl')
    joblib.dump(data['scaler_y'], f'{output_dir}scaler_y.pkl')
    
    pd.DataFrame({
        'Date': data['test_dates'],
        'Observed': data['y_test_original'],
        'Predicted': y_test_pred,
        'Residual': data['y_test_original'] - y_test_pred
    }).to_csv(f'{output_dir}lstm_test_predictions.csv', index=False, encoding='utf-8-sig')
    
    pd.DataFrame(history).to_csv(f'{output_dir}lstm_training_history.csv', index=False)
    
    pd.DataFrame({
        'Metric': ['R2', 'NSE', 'KGE', 'RMSE', 'MAE'],
        'Test': [test_metrics['R2'], test_metrics['NSE'], test_metrics['KGE'], test_metrics['RMSE'], test_metrics['MAE']],
        'Train': [train_metrics['R2'], train_metrics['NSE'], train_metrics['KGE'], train_metrics['RMSE'], train_metrics['MAE']]
    }).to_csv(f'{output_dir}lstm_metrics.csv', index=False)
    
    print(f"\nOutputs saved to {output_dir}")
    
    return model, history, test_metrics

#==============================================================================
# 8. Main
#==============================================================================
def main(train_path, test_path, output_dir='./', run_search=True):
    
    print("="*60)
    print("LSTM Training - Fast Version")
    print("="*60)
    
    print("\n[1] Loading data...")
    data = load_and_prepare_data(train_path, test_path)
    print(f"Shape: Train {data['X_train'].shape}, Test {data['X_test'].shape}")
    
    if run_search:
        print("\n[2] Hyperparameter search...")
        best_params, _ = hyperparameter_search(data, n_splits=3)
    else:
        best_params = {
            'hidden_size1': 128, 'hidden_size2': 64, 'dropout': 0.2,
            'learning_rate': 0.001, 'batch_size': 64, 'bidirectional': False
        }
        print(f"\n[2] Using default params")
    
    best_params['epochs'] = 200
    best_params['patience'] = 30
    
    print("\n[3] Training final model...")
    model, history, metrics = train_final_model(data, best_params, output_dir)
    
    print("\n" + "="*60)
    print(f"Final: R²={metrics['R2']:.4f}, NSE={metrics['NSE']:.4f}, KGE={metrics['KGE']:.4f}")
    print("="*60)
    
    return model, history, metrics

#==============================================================================
if __name__ == "__main__":
    
    TRAIN_PATH = '../数据/train_data.csv'
    TEST_PATH = '../数据/test_data.csv'
    OUTPUT_DIR = './models/lstm/'
    
    model, history, metrics = main(TRAIN_PATH, TEST_PATH, OUTPUT_DIR, run_search=True)

Using device: cuda
LSTM Training - Fast Version

[1] Loading data...
特征数量: 133
训练集样本: 7298, 测试集样本: 3287
Shape: Train (7298, 7, 19), Test (3287, 7, 19)

[2] Hyperparameter search...

Fast Hyperparameter Search (3-fold CV)

测试 6 种参数组合, 3 折CV
------------------------------------------------------------

[1/6] hidden=[64,32], dr=0.2, lr=0.001, bidir=False
    CV: KGE=0.6858, NSE=0.7296, RMSE=261.50
    *** New best KGE: 0.6858 ***

[2/6] hidden=[128,64], dr=0.2, lr=0.001, bidir=False
    CV: KGE=0.5195, NSE=0.5244, RMSE=352.04

[3/6] hidden=[128,64], dr=0.1, lr=0.001, bidir=False
    CV: KGE=0.5556, NSE=0.4436, RMSE=364.27

[4/6] hidden=[128,64], dr=0.2, lr=0.0005, bidir=False
    CV: KGE=0.7385, NSE=0.7620, RMSE=255.46
    *** New best KGE: 0.7385 ***

[5/6] hidden=[256,128], dr=0.2, lr=0.001, bidir=False
    CV: KGE=0.7169, NSE=0.7931, RMSE=251.21

[6/6] hidden=[128,64], dr=0.2, lr=0.001, bidir=True
    CV: KGE=0.4778, NSE=0.4994, RMSE=392.43

最佳参数 (KGE=0.7385):
  hidden_size1: 128
  hid